In [46]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices
from jax_sn.domain.geometric_domain import CrossSectionData

onp.set_printoptions(linewidth = 200)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
#flux_surface     =  jax_sbgeom.flux_surfaces.FluxSurface.from_hdf5("/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4")
#s_mesh           = jnp.linspace(0, 1, 10)
#tetra_mesh       =  jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_tetrahedra(flux_surface, s_mesh, ToroidalExtent.half_module(flux_surface), 50,60)
#face_template    = jsn.tools.mesh_tools._face_template(tetra_mesh[1].shape[1])
#geometric_domain = jsn.domain.GeometricDomain.from_mesh(tetra_mesh, face_template)
#quadrature_set   = jsn.quadrature_set.tn.create_tn_quadrature_set(4)
#pol_mesh         = jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(flux_surface, jnp.linspace(0,1,10)**2, 0.52,  20)
#tri_face_template = jsn.tools.mesh_tools._face_template(pol_mesh[1].shape[1])
#pol_domain        = jsn.domain.GeometricDomain.from_mesh(pol_mesh, tri_face_template)


In [48]:
from finite_elements.test_finite_elements import _get_element_jsn

In [49]:

from basix import ElementFamily, LagrangeVariant
import basix
from jax_sn.solution_domain.basix_fem import BasixLagrangianSimplex, _create_face_quadrature_on_face, _map_simplex_point_to_different_permutation, _map_face_points_to_reference_cell
from jax_sn.tools.permutation_tools import _obtain_permutation, _indices_to_permutation_number
import jax_sn
degree = 3
lasimp = BasixLagrangianSimplex(1,3)
lasimp.n_vertices_per_face(0)

3

In [50]:
from finite_elements.test_transport_operator import test_sweep_complete_vmap,  _setup_domain, _verts_conn_from_sb,  _angle_function_tn, _expected_values_from_sb, _source_from_sb, _cross_sections_from_sb, _setup_all


lagrangian_element, domain, angle, toposort, femmatrices, precomputed_data, sweep_array, cross_sections, source, expected_values = _setup_all(3, 2, _verts_conn_from_sb, _angle_function_tn, _expected_values_from_sb, _source_from_sb, _cross_sections_from_sb)

In [51]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_complete_vmap, transport_inverse_sweep_complete_vmap_fori
sweep_array_naive = transport_inverse_sweep_complete_vmap(
        femmatrices,
        domain,
        toposort,
        cross_sections, 
        precomputed_data,
        source,
        sweep_array)

In [72]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_complete_vmap, transport_inverse_sweep_complete_vmap_fori
sweep_array2 = transport_inverse_sweep_complete_vmap_fori(
        femmatrices,
        domain,
        toposort,
        cross_sections, 
        precomputed_data,
        source,
        sweep_array)
print(sweep_array2)

[[ 8.10359250e-05  8.10359250e-05 -4.88008300e-02 ...  1.29324782e-01 -4.30072232e-03  8.10359250e-05]
 [-1.06461114e-01  2.11903111e-01  2.85442900e-01 ...  1.26266686e-01  2.29366754e-01  2.15331381e-01]
 [-1.25002891e-01  1.97330294e-01  2.63917824e-01 ... -9.42848781e-03  2.34756558e-01  2.16898970e-01]
 ...
 [ 2.30357890e+00 -4.61343637e-02  6.81904916e-02 ...  2.16329448e+00  2.13440911e+00  2.26085991e+00]
 [ 3.10972497e+00  4.23796151e-02  5.24482331e-02 ...  2.26421092e+00  2.27884412e+00  2.27496759e+00]
 [ 2.41762481e+00  3.74603813e-02  6.96326163e-02 ...  2.94873740e+00  2.14468896e+00  2.23512262e+00]]


In [71]:
%timeit transport_inverse_sweep_complete_vmap_fori(femmatrices, domain, toposort, cross_sections, precomputed_data, source, sweep_array).block_until_ready()

3.02 ms ± 140 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [22]:
sweep_array2

Array([[ 8.10359250e-05,  8.10359250e-05, -4.88008300e-02, ...,  1.29324782e-01, -4.30072232e-03,  8.10359250e-05],
       [-1.06461114e-01,  2.11903111e-01,  2.85442900e-01, ...,  1.26266686e-01,  2.29366754e-01,  2.15331381e-01],
       [-1.25002891e-01,  1.97330294e-01,  2.63917824e-01, ..., -9.42848781e-03,  2.34756558e-01,  2.16898970e-01],
       ...,
       [ 2.30357890e+00, -4.61343637e-02,  6.81904916e-02, ...,  2.16329448e+00,  2.13440911e+00,  2.26085991e+00],
       [ 3.10972497e+00,  4.23796151e-02,  5.24482331e-02, ...,  2.26421092e+00,  2.27884412e+00,  2.27496759e+00],
       [ 2.41762481e+00,  3.74603813e-02,  6.96326163e-02, ...,  2.94873740e+00,  2.14468896e+00,  2.23512262e+00]], dtype=float64)

In [50]:
sweep_array

Array([[ 8.10359250e-05,  8.10359250e-05, -4.88008300e-02, ...,  1.29324782e-01, -4.30072232e-03,  8.10359250e-05],
       [-1.06461114e-01,  2.11903111e-01,  2.85442900e-01, ...,  1.26266686e-01,  2.29366754e-01,  2.15331381e-01],
       [-1.25002891e-01,  1.97330294e-01,  2.63917824e-01, ..., -9.42848781e-03,  2.34756558e-01,  2.16898970e-01],
       ...,
       [ 2.30357890e+00, -4.61343637e-02,  6.81904916e-02, ...,  2.16329448e+00,  2.13440911e+00,  2.26085991e+00],
       [ 3.10972497e+00,  4.23796151e-02,  5.24482331e-02, ...,  2.26421092e+00,  2.27884412e+00,  2.27496759e+00],
       [ 2.41762481e+00,  3.74603813e-02,  6.96326163e-02, ...,  2.94873740e+00,  2.14468896e+00,  2.23512262e+00]], dtype=float64)